# 03 — `einsum` vs `tf.matmul`

## 1 — vectors

### Create small random query, key, and value matrices

In [ ]:
# start with random tensors so the attention math is easy to inspect.
import numpy as np
import tensorflow as tf

T, S, E = 4, 4, 3                       # queries, keys, per-head width
# use a fixed seed so both implementations receive exactly the same inputs.
rng = np.random.default_rng(0)
r = lambda *s: tf.constant(rng.standard_normal(s), dtype=tf.float32)

Q, K, V = r(T, E), r(S, E), r(S, E)
# Q and K are compared to make scores; V is combined after softmax.
print("Q", tuple(Q.shape), " K", tuple(K.shape), " V", tuple(V.shape))

### Compute attention logits two equivalent ways

Compare matrix multiplication with an explicit `einsum` dot product.

In [ ]:
# calculate the same query-key scores in two different ways.
# matrix multiplication computes all pairwise dot products at once.
# einsum writes the shared embedding dimension explicitly.
logits_mm = tf.matmul(Q, K, transpose_b=True)          # (T,E) x (E,S)
logits_es = tf.einsum("te,se->ts", Q, K)

np.testing.assert_allclose(logits_mm, logits_es, rtol=1e-5, atol=1e-6)
print("logits", tuple(logits_mm.shape),
      " max|diff|", float(np.max(np.abs(logits_mm - logits_es))))

### Convert logits to attention weights and compute context

Scale the scores, apply softmax, and combine the value vectors.

In [ ]:
# turn the scores into attention weights, then use them to mix the values.
# softmax makes each row sum to one, so the result is a weighted average.
a_mm = tf.nn.softmax(logits_mm / np.sqrt(E), axis=-1)
a_es = tf.nn.softmax(logits_es / np.sqrt(E), axis=-1)

ctx_mm = tf.matmul(a_mm, V)                            # (T,S) x (S,E)
ctx_es = tf.einsum("ts,se->te", a_es, V)

np.testing.assert_allclose(ctx_mm, ctx_es, rtol=1e-5, atol=1e-6)
print("context", tuple(ctx_mm.shape),
      " max|diff|", float(np.max(np.abs(ctx_mm - ctx_es))))

### Count MACs and estimated FLOPs for the toy example

In [ ]:
def einsum_macs(equation, *shapes, batch_axis=""):
    # count one multiply-accumulate for each combination of indices.
    lhs, out = equation.split("->")
    dims = {}
    for spec, shape in zip(lhs.split(","), shapes):
        assert len(spec) == len(shape), f"{spec} vs {tuple(shape)}"
        for ch, n in zip(spec, shape):
            assert dims.setdefault(ch, int(n)) == int(n), f"index '{ch}' mismatch"
    macs = 1
    for ch in (set(lhs) - {","}) | set(out):
        if ch != batch_axis:
            macs *= dims[ch]
    return macs

# check the einsum count against the matrix-multiplication count by hand.
for name, eq, shapes, by_hand in [
    ("QK^T",   "te,se->ts", [(T, E), (S, E)], T * E * S),
    ("attn.V", "ts,se->te", [(T, S), (S, E)], T * S * E),
]:
    m = einsum_macs(eq, *shapes)
    assert m == by_hand, f"{name}: einsum {m} != matmul {by_hand}"
    print(f"{name:7s} {eq:12s} einsum {m:5d}  matmul {by_hand:5d}  FLOPs {2 * m:5d}")

## 2 — the same checks at `qat.py` shapes

### Load the model architecture dimensions

In [ ]:
import os
from keras.layers import Dense, EinsumDense

CODE = os.environ["PYTHONPATH"]
from bnhgq2 import config as bncfg
from bnhgq2.qat import bitnet_binary_ste

# load the architecture so the test uses the same dimensions as qat.py.
cfg = bncfg.load_config(os.path.join(CODE, "configs", "r14-l1x3-n8-w1a8.json"))
A = cfg["arch"]

# keep the batch size small while preserving the model's real tensor shapes.
B   = 4
T   = int(A["n_part"])
F   = int(A["n_feat"])
D   = int(A["d_model"])
H   = int(A["n_heads"])
E   = D // H
FFN = int(A["ffn_dim"])
assert H * E == D

print(cfg["name"], "-> T", T, "| F", F, "| D", D, "| H", H, "| E", E, "| FFN", FFN)

### Create random inputs, weights, and an equality-check helper

In [ ]:
# generate reproducible float32 test tensors from the shared random generator.
def rand(*shape):
    return tf.constant(rng.standard_normal(shape), dtype=tf.float32)

# compare two implementations and report the largest numerical difference.
def same(name, a, b):
    a, b = np.asarray(a), np.asarray(b)
    assert a.shape == b.shape, f"{name}: {a.shape} vs {b.shape}"
    np.testing.assert_allclose(a, b, rtol=1e-5, atol=1e-6)
    print(f"  OK  {name:22s} {str(tuple(a.shape)):18s}"
          f" max|diff| {float(np.max(np.abs(a - b))):.3e}")

# x is the input sequence; the four weight tensors are used by attention.
x  = rand(B, T, D)
Wq = rand(D, H, E)
Wk = rand(D, H, E)
Wv = rand(D, H, E)
Wo = rand(H, E, D)

### Project the input into queries, keys, and values

Verify `einsum` and reshaped matrix multiplication produce identical Q, K, and V tensors.

In [ ]:
# project each token into queries, keys, and values for every head.
# the shared d index is summed over during each projection.
q_e = tf.einsum("btd,dhe->bthe", x, Wq)
k_e = tf.einsum("btd,dhe->bthe", x, Wk)
v_e = tf.einsum("btd,dhe->bthe", x, Wv)

def proj_matmul(x, W):
    Dw, Hw, Ew = W.shape
    # flatten the head dimensions so a regular matmul can do the projection.
    flat = tf.matmul(x, tf.reshape(W, (Dw, Hw * Ew)))      # (B,T,D) x (D,H*E)
    # put the head dimensions back after the matmul.
    return tf.reshape(flat, (-1, x.shape[1], Hw, Ew))      # -> (B,T,H,E)

q_m, k_m, v_m = proj_matmul(x, Wq), proj_matmul(x, Wk), proj_matmul(x, Wv)
same("Q", q_e, q_m)
same("K", k_e, k_m)
same("V", v_e, v_m)

### Compute attention scores and softmax weights

Transpose the head and sequence dimensions for batched matrix multiplication, then compare it with `einsum`.

In [ ]:
# compare every query with every key to get the attention logits.
# each score is a dot product over the per-head embedding dimension.
scores_e = tf.einsum("bthe,bshe->bhts", q_e, k_e)

qh = tf.transpose(q_m, [0, 2, 1, 3])                       # (B,T,H,E) -> (B,H,T,E)
kh = tf.transpose(k_m, [0, 2, 1, 3])
scores_m = tf.matmul(qh, kh, transpose_b=True)             # -> (B,H,T,S)
same("QK^T", scores_e, scores_m)

# scale before softmax so the scores stay well behaved as E grows.
scale  = 1.0 / np.sqrt(E)
attn_e = tf.nn.softmax(scores_e * scale, axis=-1)
attn_m = tf.nn.softmax(scores_m * scale, axis=-1)
same("softmax(QK^T/sqrt E)", attn_e, attn_m)

### Compute context and project the heads back to model width

Combine values using the attention weights, then apply the output projection `W_o`.

In [ ]:
# use the attention weights to make a weighted combination of the values.
# transpose the values so their dimensions line up with the batched matmul.
ctx_e = tf.einsum("bhts,bshe->bthe", attn_e, v_e)

vh    = tf.transpose(v_m, [0, 2, 1, 3])                    # (B,S,H,E) -> (B,H,S,E)
ctx_m = tf.transpose(tf.matmul(attn_m, vh), [0, 2, 1, 3])  # (B,H,T,E) -> (B,T,H,E)
same("attn . V", ctx_e, ctx_m)

# combine the heads back into the original model dimension.
out_e = tf.einsum("bthe,hed->btd", ctx_e, Wo)
out_m = tf.matmul(tf.reshape(ctx_m, (-1, T, H * E)), tf.reshape(Wo, (H * E, D)))
same("W_o", out_e, out_m)

### Repeat the projection check with a binary weight kernel

In [ ]:
# repeat the projection check after replacing the weights with binary values.
# the binary kernel should contain only the intended {-1, +1} values.
Wq_b = bitnet_binary_ste(Wq)
print("distinct Wq values:", np.unique(np.round(np.asarray(Wq_b), 8)))
same("Q (binary W)", tf.einsum("btd,dhe->bthe", x, Wq_b), proj_matmul(x, Wq_b))

## 3 — parameter counts

`EinsumDense` vs `Dense`: this section checks how many stored weights each layer
creates.

In [ ]:
# EinsumDense vs Dense, identical weights
ein = EinsumDense("btd,dhe->bthe", output_shape=(T, H, E), bias_axes=None)
ein.build((None, T, D))
ein.set_weights([np.asarray(Wq)])

den = Dense(H * E, use_bias=False)
den.build((None, T, D))
den.set_weights([np.asarray(tf.reshape(Wq, (D, H * E)))])

same("layer output", ein(x), tf.reshape(den(x), (-1, T, H, E)))
print("  EinsumDense kernel", tuple(ein.kernel.shape),
      "->", int(np.prod(ein.kernel.shape)), "params")
print("  Dense       kernel", tuple(den.kernel.shape),
      "->", int(np.prod(den.kernel.shape)), "params")

In [ ]:
def ed_params(equation, output_shape, input_shape, bias_axes=None):
    lyr = EinsumDense(equation, output_shape=output_shape, bias_axes=bias_axes)
    lyr.build(input_shape)
    k = int(np.prod(lyr.kernel.shape))
    b = int(np.prod(lyr.bias.shape)) if lyr.bias is not None else 0
    bs = tuple(lyr.bias.shape) if lyr.bias is not None else None
    return tuple(lyr.kernel.shape), k, bs, b

# Same output shape, same MACs, different parameter count. This is the failure mode:
# EinsumDense sizes its kernel from the WEIGHT subscript, so a stray 't' costs T x.
for label, eq, bax in [
    ("correct",          "btd,dhe->bthe",  None),
    ("t in weight spec", "btd,tdhe->bthe", None),
    ("t in bias spec",   "btd,dhe->bthe",  "the"),
]:
    ks, k, bs, b = ed_params(eq, (T, H, E), (None, T, D), bax)
    print(f"  {label:18s} {eq:16s} kernel {str(ks):16s}"
          f" bias {str(bs):12s} total {k + b:6d}")

base = ed_params("btd,dhe->bthe", (T, H, E), (None, T, D))[1]
bad  = ed_params("btd,tdhe->bthe", (T, H, E), (None, T, D))[1]
print(f"  wrong weight spec costs {bad // base}x the weights ({base} -> {bad})")

In [ ]:
# the built model, not standalone layers
from bnhgq2 import qat

model, taps = qat.build_qat_model(cfg, seed=1)

hdr = f"{'layer':30s} {'equation':16s} {'kernel':14s} {'K':>6s} {'bias':>10s} {'B':>5s}"
print(hdr)
print("-" * len(hdr))
for lyr in model.layers:
    k = getattr(lyr, "_kernel", None)
    if k is None:
        continue
    b  = getattr(lyr, "bias", None)
    nb = int(np.prod(b.shape)) if b is not None else 0
    print(f"{lyr.name:30s} {getattr(lyr, 'equation', 'matmul (QDense)'):16s}"
          f" {str(tuple(k.shape)):14s} {int(np.prod(k.shape)):6d}"
          f" {str(tuple(b.shape)) if b is not None else '-':>10s} {nb:5d}")

In [ ]:
wq = model.get_layer("bit_block_0_attn_Wq")
print("class    :", type(wq).__name__)
print("equation :", wq.equation)
print("kernel   :", tuple(wq._kernel.shape), " expected", (D, H, E))
print("bias     :", wq.bias)
assert tuple(wq._kernel.shape) == (D, H, E), "Wq kernel is not (D,H,E)!"
assert wq.bias is None, "Wq grew a bias"
print(f"=> {D * H * E} weights per projection, not {T * D * H * E}")

## 4 — MACs / FLOPs at `qat.py` shapes

In [ ]:
ROWS = [
    ("Wq projection", "btd,dhe->bthe",   [(B,T,D), (D,H,E)],     T*D*H*E, "(T x D) x (D x HE)",      "weighted"),
    ("Wk projection", "btd,dhe->bthe",   [(B,T,D), (D,H,E)],     T*D*H*E, "(T x D) x (D x HE)",      "weighted"),
    ("Wv projection", "btd,dhe->bthe",   [(B,T,D), (D,H,E)],     T*D*H*E, "(T x D) x (D x HE)",      "weighted"),
    ("QK^T logits",   "bthe,bshe->bhts", [(B,T,H,E), (B,T,H,E)], H*T*T*E, "H x [(T x E) x (E x S)]", "act x act"),
    ("attn . V",      "bhts,bshe->bthe", [(B,H,T,T), (B,T,H,E)], H*T*T*E, "H x [(T x S) x (S x E)]", "act x act"),
    ("Wo projection", "bthe,hed->btd",   [(B,T,H,E), (H,E,D)],   T*H*E*D, "(T x HE) x (HE x D)",     "weighted"),
]

hdr = f"{'op':14s} {'equation':18s} {'einsum':>9s} {'matmul':>9s}  {'matmul spelling':26s} kind"
print(hdr)
print("-" * len(hdr))
tot = wgt = act = 0
for name, eq, shapes, by_hand, form, kind in ROWS:
    m = einsum_macs(eq, *shapes, batch_axis="b")
    assert m == by_hand, f"{name}: einsum {m} != matmul {by_hand}"
    tot += m
    wgt += m if kind == "weighted" else 0
    act += m if kind != "weighted" else 0
    print(f"{name:14s} {eq:18s} {m:9,d} {by_hand:9,d}  {form:26s} {kind}")
print("-" * len(hdr))
print(f"  attention block: {tot:,} MACs/jet = {2 * tot:,} FLOPs/jet")
print(f"  weighted contractions   : {wgt:6,d} MACs/jet  <- binary kernels")
print(f"  activation x activation : {act:6,d} MACs/jet  <- no stored weights")